# 03 — Baseline Models

Trains a few standard classifiers (Logistic Regression, KNN, Decision Tree) with **default hyperparameters** to establish a performance baseline before any tuning or resampling. This tells us how much headroom there is, and which model families are worth tuning further.

Loads the train/test split and preprocessor saved by [02_feature_engineering.ipynb](02_feature_engineering.ipynb).

In [1]:
import sys
sys.path.append('..')

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=UserWarning)
warnings.simplefilter("ignore", category=ConvergenceWarning)

import joblib
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from src import modeling

## Load artifacts from 02_feature_engineering

In [2]:
PROCESSED_DIR = '../data/processed'
MODELS_DIR = '../models'

X_train = pd.read_parquet(f'{PROCESSED_DIR}/X_train.parquet')
X_test = pd.read_parquet(f'{PROCESSED_DIR}/X_test.parquet')
y_train = pd.read_parquet(f'{PROCESSED_DIR}/y_train.parquet')['readmitted_30']
y_test = pd.read_parquet(f'{PROCESSED_DIR}/y_test.parquet')['readmitted_30']

preprocessor = joblib.load(f'{MODELS_DIR}/preprocessor.joblib')

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

X_train: (80091, 29), X_test: (20023, 29)


## Baseline models

In [3]:
default_models = {
    'logisticRegression': LogisticRegression(max_iter=1000),
    'knn': KNeighborsClassifier(),
    'decisiontreeclassifier': DecisionTreeClassifier()
}

In [4]:
results_score1 = modeling.get_baseline_model_stats(
    default_models, preprocessor, X_train, y_train, X_test, y_test
)
results_score1_df = pd.DataFrame(results_score1)
results_score1_df

,Model,Train time,Train accuracy,Test accuracy,Accuracy,Precision,Recall,F1 Score
0,logisticRegression,1.242760,0.886529,0.886381,0.886381,0.472973,0.015412,0.029851
1,knn,0.282252,0.893521,0.878440,0.878440,0.304556,0.055923,0.094494
2,decisiontreeclassifier,1.199777,1.000000,0.802927,0.802927,0.165935,0.183179,0.174131


#### Finding
- All models seem to have a high accuracy but low on precision and recall. Accuracy is high since models are good at
  predicting the majority class which is 88% of data
- Logistic regression shows overall stability with accuracies, precision is higher than decision tree classifier.
  Recall of 1.5% shows that model predicts on 2 correct positive class out of 100 which is very low.
- KNN - F1 score is still low, with recall slightly better than logistic regression
- Decision Tree Classifier - although high on train accuracy, accuracy drops for test data showing overfitting.
  It has better recall 18% than Logistic regression, but it is still low in predicting positive class.
- Root cause is highly imbalanced target class <30 days, 88% of the data is negative class.

## Persist baseline results

In [5]:
results_score1_df.to_csv('../data/processed/baseline_model_results.csv', index=False)
print("Saved baseline results to data/processed/baseline_model_results.csv")

Saved baseline results to data/processed/baseline_model_results.csv


**Next**: [04_model_tuning.ipynb](04_model_tuning.ipynb) addresses the class imbalance via SMOTE / undersampling and hyperparameter tuning to improve on these baselines.